# Сравнение качества ASR-моделей на собственном датасете

Этот ноутбук предназначен для воспроизводимого сравнения качества моделей распознавания речи в текст на вашем CSV-датасете.

### Что умеет ноутбук
- загружает CSV с разметкой и аудиопутями;
- нормализует пути через `PATH_PREFIX_MAP` для разных окружений;
- чистит референсную транскрипцию от таймкодов, меток спикеров и служебных вставок;
- считает `WER`, `CER`, среднюю/медианную ошибку по строкам, суммарную задержку и `RTF`;
- сохраняет лидерборд и предсказания каждой модели;
- позволяет удобно добавлять новые модели через единый `MODEL_REGISTRY`.

### Модели в сравнении
- `openai/whisper-large-v3-turbo`
- `bond005/whisper-podlodka-turbo`
- `jonatasgrosman/wav2vec2-large-xlsr-53-russian`
- `nvidia/stt_ru_fastconformer_hybrid_large_pc`
- `t-tech/T-one`
- `ai-sage/GigaAM-v3` в режимах `ctc` и `rnnt`

> Ноутбук сделан так, чтобы модели загружались лениво: в память попадает только та модель, которая сейчас оценивается.


## 1. Установка зависимостей

> Для `T-one` комфортнее Linux / WSL / Kaggle.  
> Если окружение уже подготовлено, эту ячейку можно пропустить.


In [ ]:
# %pip install -q -U jiwer

In [ ]:
# Базовые зависимости
# %pip install -q -U transformers datasets[audio] accelerate jiwer evaluate librosa soundfile sentencepiece hydra-core omegaconf

# T-one (Python API)
# %pip install -q git+https://github.com/voicekit-team/T-one.git

# Опционально: long-form зависимости для GigaAM, если понадобятся внешние пайплайны сегментации
# %pip install -q pyannote.audio torchcodec

# NVIDIA NeMo для FastConformer Hybrid
# %pip install -q -U nemo_toolkit['asr']

## 2. Импорты


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import shutil
import tempfile
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import display
from jiwer import cer, process_words, wer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

## 3. Конфиг эксперимента

Главная идея: все параметры собраны в одном месте.  
Здесь же находится `PATH_PREFIX_MAP`, который можно менять под новое окружение.


In [ ]:
# -----------------------------
# Пути и источники данных
# -----------------------------
DATA_CSV_CANDIDATES = [
    # "/kaggle/input/datasets/coreytaz332/markup-calls/_updated.csv",
]

# Можно задать конкретный путь вручную:
# DATA_CSV_PATH = "/kaggle/input/your-dataset/Выгрузка_updated.csv"
DATA_CSV_PATH = "node/Выгрузка_updated.csv"

# -----------------------------
# Маппинг путей под другое окружение
# -----------------------------
PATH_PREFIX_MAP = {
    # r"Z:\calls2": "/kaggle/input/datasets/coreytaz332/markup-calls/calls2/calls2",
}

# Пример добавления новых маппингов:
# PATH_PREFIX_MAP.update({
#     r"D:\audio": "/kaggle/input/my-audio/audio",
#     r"\\server\share\calls": "/mnt/storage/calls",
# })

# -----------------------------
# Названия колонок в CSV
# -----------------------------
REFERENCE_COLUMN = "transcription"
ABSOLUTE_PATH_COLUMN = "absolute_path"
ID_COLUMN = "call_id"

# -----------------------------
# Настройки выборки
# -----------------------------
USE_ONLY_EXISTING_AUDIO = True
DROP_EMPTY_REFERENCES = True
MAX_SAMPLES = 1  # None -> взять весь датасет
RANDOM_SEED = 42
SHUFFLE_BEFORE_SAMPLE = True

# -----------------------------
# Настройки вычислений
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEFAULT_TARGET_SR = 16_000

# -----------------------------
# Куда сохранять результаты
# -----------------------------
RUN_NAME = time.strftime("asr_benchmark_%Y%m%d_%H%M%S")
ARTIFACTS_DIR = Path("kaggle/working/artifacts") / RUN_NAME
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(f"DEVICE        : {DEVICE}")
print(f"TORCH_DTYPE   : {TORCH_DTYPE}")
print(f"ARTIFACTS_DIR : {ARTIFACTS_DIR.resolve()}")

## 4. Утилиты

Здесь находится вся "грязная" инфраструктура:
- поиск CSV;
- перенос путей между Windows / Linux / Kaggle;
- очистка и нормализация текста;
- получение длительности аудио;
- подготовка временных чанков для моделей, которым неудобно работать с длинным аудио.


In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(RANDOM_SEED)


def locate_csv(explicit_path: Optional[str], candidates: Sequence[str]) -> Path:
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f"DATA_CSV_PATH не найден: {path}")
        return path

    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path

    checked = [str(Path(x).resolve()) for x in candidates]
    raise FileNotFoundError(
        "Не удалось найти CSV-файл. Проверьте DATA_CSV_PATH или DATA_CSV_CANDIDATES:\\n"
        + "\\n".join(checked)
    )


def normalize_path_string(path_value: Any) -> str:
    if path_value is None or (isinstance(path_value, float) and np.isnan(path_value)):
        return ""
    return str(path_value).strip().replace("\\\\", "/")


def normalize_raw_path(x):
    if pd.isna(x):
        return None
    x = str(x).strip().strip('"').strip("'")
    x = x.replace("/", "\\")
    x = re.sub(r"\\+", r"\\", x)
    return x.rstrip("\\")


def apply_path_prefix_map(path_value: Any, path_prefix_map: Dict[str, str]) -> str:
    raw_path = normalize_path_string(path_value)
    if not raw_path:
        return ""

    raw_norm = raw_path.replace("\\\\", "/")
    raw_fold = raw_norm.casefold()

    for src_prefix, dst_prefix in path_prefix_map.items():
        src_norm = normalize_path_string(src_prefix)
        src_fold = src_norm.casefold()
        if raw_fold.startswith(src_fold):
            suffix = raw_norm[len(src_norm) :].lstrip("/")
            return str(Path(dst_prefix) / suffix)

    return raw_norm


def resolve_audio_path(raw_path, path_prefix_map=None):
    raw_path = normalize_raw_path(raw_path)
    if raw_path is None:
        return None

    path_prefix_map = path_prefix_map or {}
    raw_lower = raw_path.lower()

    for src_prefix, dst_prefix in path_prefix_map.items():
        src_prefix_norm = normalize_raw_path(src_prefix)
        src_lower = src_prefix_norm.lower()

        if raw_lower == src_lower or raw_lower.startswith(src_lower + "\\"):
            relative_part = raw_path[len(src_prefix_norm) :]
            relative_part = relative_part.lstrip("\\/")
            relative_part = relative_part.replace("\\", "/")
            return dst_prefix.rstrip("/") + (
                "/" + relative_part if relative_part else ""
            )

    if raw_path.startswith("/"):
        return raw_path

    return raw_path.replace("\\", "/")


def file_exists(raw_path, path_prefix_map=None):
    resolved = resolve_audio_path(raw_path, path_prefix_map)
    if resolved is None or pd.isna(resolved):
        return False
    return Path(resolved).exists()


def get_audio_duration_s(audio_path: str) -> Optional[float]:
    try:
        info = sf.info(audio_path)
        return float(info.duration)
    except Exception:
        try:
            return float(librosa.get_duration(path=audio_path))
        except Exception:
            return None


TIMECODE_RE = re.compile(r"\(\d{2}:\d{2}(?::\d{2})?\)")
SPEAKER_RE = re.compile(r"\bспикер\s*\d+\s*:\s*", flags=re.IGNORECASE)
BRACKETS_RE = re.compile(r"\[[^\]]+\]")
MULTISPACE_RE = re.compile(r"\s+")


def strip_transcription_markup(text: Any) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""

    text = str(text)
    text = TIMECODE_RE.sub(" ", text)
    text = SPEAKER_RE.sub(" ", text)
    text = BRACKETS_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


def normalize_text_for_metrics(text: Any) -> str:
    text = strip_transcription_markup(text)
    text = text.lower().replace("ё", "е")
    text = re.sub(r"[^0-9a-zа-я\s]", " ", text, flags=re.IGNORECASE)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


def safe_wer(ref: str, hyp: str) -> float:
    ref = ref or ""
    hyp = hyp or ""
    if not ref and not hyp:
        return 0.0
    if not ref and hyp:
        return 1.0
    return float(wer(ref, hyp))


def safe_cer(ref: str, hyp: str) -> float:
    ref = ref or ""
    hyp = hyp or ""
    if not ref and not hyp:
        return 0.0
    if not ref and hyp:
        return 1.0
    return float(cer(ref, hyp))


def coerce_transcription_output(output: Any) -> str:
    if output is None:
        return ""

    if isinstance(output, str):
        return output.strip()

    if isinstance(output, dict):
        for key in ("text", "transcription", "prediction", "pred_text"):
            if key in output and output[key] is not None:
                return str(output[key]).strip()

        # Иногда remote code возвращает более сложные словари.
        # Попробуем аккуратно вытащить текстовые поля рекурсивно.
        parts = []
        for value in output.values():
            text = coerce_transcription_output(value)
            if text:
                parts.append(text)
        return " ".join(parts).strip()

    if isinstance(output, (list, tuple)):
        parts = []
        for item in output:
            if hasattr(item, "text"):
                parts.append(str(item.text).strip())
            else:
                text = coerce_transcription_output(item)
                if text:
                    parts.append(text)
        return " ".join(parts).strip()

    if hasattr(output, "text"):
        return str(output.text).strip()

    return str(output).strip()


def iter_audio_chunks_to_tempfiles(
    audio_path: str,
    chunk_length_s: float = 22.0,
    overlap_s: float = 0.25,
    target_sr: int = DEFAULT_TARGET_SR,
) -> Iterator[str]:
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    chunk_size = int(chunk_length_s * sr)
    step_size = int(max(chunk_length_s - overlap_s, 1.0) * sr)

    with tempfile.TemporaryDirectory(prefix="asr_chunks_") as tmp_dir:
        for chunk_idx, start in enumerate(range(0, len(audio), step_size)):
            end = min(start + chunk_size, len(audio))
            if end <= start:
                continue

            chunk = audio[start:end]
            chunk_path = Path(tmp_dir) / f"chunk_{chunk_idx:04d}.wav"
            sf.write(chunk_path, chunk, sr)

            yield str(chunk_path)

            if end >= len(audio):
                break


def maybe_cleanup_torch() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Загрузка и подготовка датасета


In [ ]:
csv_path = locate_csv(DATA_CSV_PATH, DATA_CSV_CANDIDATES)
print(f"Используем CSV: {csv_path.resolve()}")

raw_df = pd.read_csv(csv_path)
print(raw_df.shape)
display(raw_df.head(3))

In [ ]:
required_columns = [REFERENCE_COLUMN, ABSOLUTE_PATH_COLUMN]
missing_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_columns:
    raise KeyError(f"В CSV отсутствуют обязательные колонки: {missing_columns}")

df = raw_df.copy()

if ID_COLUMN not in df.columns:
    df[ID_COLUMN] = np.arange(len(df))

df["audio_path"] = df[ABSOLUTE_PATH_COLUMN].apply(
    lambda x: str(resolve_audio_path(x, PATH_PREFIX_MAP))
)
df["audio_exists"] = df[ABSOLUTE_PATH_COLUMN].apply(
    lambda x: file_exists(x, PATH_PREFIX_MAP)
)
df["reference_raw"] = df[REFERENCE_COLUMN].fillna("").astype(str)
df["reference_clean"] = df["reference_raw"].apply(strip_transcription_markup)
df["reference_norm"] = df["reference_raw"].apply(normalize_text_for_metrics)

if DROP_EMPTY_REFERENCES:
    df = df[df["reference_norm"].astype(bool)].copy()

if USE_ONLY_EXISTING_AUDIO:
    df = df[df["audio_exists"]].copy()

if SHUFFLE_BEFORE_SAMPLE:
    df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

if MAX_SAMPLES is not None:
    df = df.head(min(MAX_SAMPLES, len(df))).copy()

df["audio_duration_s"] = df["audio_path"].apply(get_audio_duration_s)

print(f"Итоговая выборка для эксперимента: {len(df)}")
display(df)

## 6. Реестр моделей и адаптеры

Ниже сделана расширяемая схема:

- `MODEL_REGISTRY` — список моделей и их параметров;
- `BaseASRAdapter` — общий интерфейс;
- `TransformersPipelineAdapter` — универсальный адаптер для моделей Hugging Face, которые хорошо работают через `pipeline`;
- `TOneAdapter`, `GigaAMAdapter` и `NeMoASRAdapter` — кастомные адаптеры для моделей со своей логикой.

Чтобы добавить новую модель, чаще всего достаточно просто вписать ещё один элемент в `MODEL_REGISTRY`.


In [ ]:
@dataclass
class ModelSpec:
    key: str
    label: str
    adapter_cls: type
    init_kwargs: Dict[str, Any] = field(default_factory=dict)
    enabled: bool = True


class BaseASRAdapter:
    def __init__(self, spec: ModelSpec):
        self.spec = spec
        self.device = DEVICE
        self.dtype = TORCH_DTYPE

    def load(self) -> None:
        raise NotImplementedError

    def transcribe(self, audio_path: str) -> str:
        raise NotImplementedError

    def unload(self) -> None:
        maybe_cleanup_torch()


class TransformersPipelineAdapter(BaseASRAdapter):
    def load(self) -> None:
        from transformers import (
            AutoModelForCTC,
            AutoModelForSpeechSeq2Seq,
            AutoProcessor,
            Wav2Vec2Processor,
            pipeline,
        )

        model_id = self.spec.init_kwargs["model_id"]
        model_kind = self.spec.init_kwargs.get("kind", "ctc")
        low_cpu_mem_usage = self.spec.init_kwargs.get("low_cpu_mem_usage", True)

        if model_id == "jonatasgrosman/wav2vec2-large-xlsr-53-russian":
            processor = Wav2Vec2Processor.from_pretrained(model_id)
        else:
            processor = AutoProcessor.from_pretrained(model_id)

        if model_kind == "seq2seq":
            model = AutoModelForSpeechSeq2Seq.from_pretrained(
                model_id,
                dtype=self.dtype,
                low_cpu_mem_usage=low_cpu_mem_usage,
            )
        elif model_kind == "ctc":
            model = AutoModelForCTC.from_pretrained(model_id)
        else:
            raise ValueError(f"Неизвестный kind={model_kind!r}")

        if self.device == "cuda":
            model = model.to("cuda")

        pipe_kwargs = {
            "task": "automatic-speech-recognition",
            "model": model,
            "tokenizer": getattr(processor, "tokenizer", None),
            "feature_extractor": getattr(processor, "feature_extractor", None),
            "device": 0 if self.device == "cuda" else -1,
        }

        if model_kind == "seq2seq":
            pipe_kwargs["dtype"] = self.dtype

        for optional_key in ("chunk_length_s", "stride_length_s", "batch_size"):
            if optional_key in self.spec.init_kwargs:
                pipe_kwargs[optional_key] = self.spec.init_kwargs[optional_key]

        self.pipe = pipeline(**pipe_kwargs)

    def transcribe(self, audio_path: str) -> str:
        call_kwargs = {}

        if "generate_kwargs" in self.spec.init_kwargs:
            call_kwargs["generate_kwargs"] = self.spec.init_kwargs["generate_kwargs"]

        if "return_timestamps" in self.spec.init_kwargs:
            call_kwargs["return_timestamps"] = self.spec.init_kwargs[
                "return_timestamps"
            ]

        result = self.pipe(audio_path, **call_kwargs)
        print(result)
        return coerce_transcription_output(result)

    def unload(self) -> None:
        self.pipe = None
        super().unload()


# class TOneAdapter(BaseASRAdapter):
#     def load(self) -> None:
#         from tone import StreamingCTCPipeline
#
#         self.pipeline = StreamingCTCPipeline.from_hugging_face()
#
#     def transcribe(self, audio_path: str) -> str:
#         from tone import read_audio
#
#         audio = read_audio(audio_path)
#         result = self.pipeline.forward_offline(audio)
#         return coerce_transcription_output(result)
#
#     def unload(self) -> None:
#         self.pipeline = None
#         super().unload()


class GigaAMAdapter(BaseASRAdapter):
    def load(self) -> None:
        from transformers import AutoModel

        revision = self.spec.init_kwargs["revision"]
        self.model = AutoModel.from_pretrained(
            "ai-sage/GigaAM-v3",
            revision=revision,
            trust_remote_code=True,
        )
        self.model = self.model.to(self.device)

    def _transcribe_single_file(self, audio_path: str) -> str:
        output = self.model.transcribe(audio_path)
        print(output)
        return coerce_transcription_output(output)

    def transcribe(self, audio_path: str) -> str:
        duration_s = get_audio_duration_s(audio_path)
        long_audio_threshold_s = self.spec.init_kwargs.get(
            "long_audio_threshold_s", 24.5
        )
        chunk_long_audio = self.spec.init_kwargs.get("chunk_long_audio", True)

        if duration_s and duration_s > long_audio_threshold_s and chunk_long_audio:
            parts = []
            for chunk_path in iter_audio_chunks_to_tempfiles(
                audio_path=audio_path,
                chunk_length_s=self.spec.init_kwargs.get("chunk_length_s", 22.0),
                overlap_s=self.spec.init_kwargs.get("overlap_s", 0.25),
                target_sr=self.spec.init_kwargs.get("target_sr", DEFAULT_TARGET_SR),
            ):
                parts.append(self._transcribe_single_file(chunk_path))
            return " ".join([part for part in parts if part]).strip()

        return self._transcribe_single_file(audio_path)

    def unload(self) -> None:
        self.model = None
        super().unload()


class NeMoASRAdapter(BaseASRAdapter):
    def load(self) -> None:
        import nemo.collections.asr as nemo_asr

        model_id = self.spec.init_kwargs["model_id"]
        self.model = nemo_asr.models.EncDecHybridRNNTCTCBPEModel.from_pretrained(
            model_name=model_id
        )

        if self.device == "cuda":
            self.model = self.model.to("cuda")

        decoder_type = self.spec.init_kwargs.get("decoder_type")
        if decoder_type:
            self.model.change_decoding_strategy(decoder_type=decoder_type)

    def _extract_text(self, output: Any) -> str:
        if isinstance(output, str):
            return output
        text = getattr(output, "text", None)
        if isinstance(text, str):
            return text
        if isinstance(output, dict):
            return coerce_transcription_output(output)
        return str(output)

    def _transcribe_single_file(self, audio_path: str) -> str:
        outputs = self.model.transcribe([audio_path])
        print(outputs)
        if isinstance(outputs, (list, tuple)) and outputs:
            return self._extract_text(outputs[0]).strip()
        return self._extract_text(outputs).strip()

    def transcribe(self, audio_path: str) -> str:
        duration_s = get_audio_duration_s(audio_path)
        long_audio_threshold_s = self.spec.init_kwargs.get(
            "long_audio_threshold_s", 24.5
        )
        chunk_long_audio = self.spec.init_kwargs.get("chunk_long_audio", True)

        if duration_s and duration_s > long_audio_threshold_s and chunk_long_audio:
            parts = []
            for chunk_path in iter_audio_chunks_to_tempfiles(
                audio_path=audio_path,
                chunk_length_s=self.spec.init_kwargs.get("chunk_length_s", 22.0),
                overlap_s=self.spec.init_kwargs.get("overlap_s", 0.25),
                target_sr=self.spec.init_kwargs.get("target_sr", DEFAULT_TARGET_SR),
            ):
                parts.append(self._transcribe_single_file(chunk_path))
            return " ".join([part for part in parts if part]).strip()

        return self._transcribe_single_file(audio_path)

    def unload(self) -> None:
        self.model = None
        super().unload()


MODEL_REGISTRY: Dict[str, ModelSpec] = {
    "whisper_large_v3_turbo": ModelSpec(
        key="whisper_large_v3_turbo",
        label="Whisper Large v3 Turbo",
        adapter_cls=TransformersPipelineAdapter,
        init_kwargs={
            "model_id": "openai/whisper-large-v3-turbo",
            "kind": "seq2seq",
            "chunk_length_s": 30,
            "batch_size": 8,
            "generate_kwargs": {
                "task": "transcribe",
                "language": "russian",
            },
        },
        enabled=True,
    ),
    "whisper_podlodka_turbo": ModelSpec(
        key="whisper_podlodka_turbo",
        label="Whisper Podlodka Turbo",
        adapter_cls=TransformersPipelineAdapter,
        init_kwargs={
            "model_id": "bond005/whisper-podlodka-turbo",
            "kind": "seq2seq",
            "chunk_length_s": 30,
            "batch_size": 8,
            "generate_kwargs": {
                "task": "transcribe",
                "language": "russian",
            },
        },
        enabled=True,
    ),
    "whisper_large_v3": ModelSpec(
        key="whisper_large_v3",
        label="Whisper Large v3",
        adapter_cls=TransformersPipelineAdapter,
        init_kwargs={
            "model_id": "openai/whisper-large-v3",
            "kind": "seq2seq",
            "chunk_length_s": 30,
            "batch_size": 8,
            "generate_kwargs": {
                "task": "transcribe",
                "language": "russian",
            },
        },
        enabled=True,
    ),
    "wav2vec2_xlsr_53_ru": ModelSpec(
        key="wav2vec2_xlsr_53_ru",
        label="Wav2Vec2 XLSR-53 Russian",
        adapter_cls=TransformersPipelineAdapter,
        init_kwargs={
            "model_id": "jonatasgrosman/wav2vec2-large-xlsr-53-russian",
            "kind": "ctc",
            "chunk_length_s": 20,
            "stride_length_s": (4, 2),
            "batch_size": 8,
        },
        enabled=True,
    ),
    "nvidia_fastconformer_hybrid_large_pc_rnnt": ModelSpec(
        key="nvidia_fastconformer_hybrid_large_pc_rnnt",
        label="NVIDIA FastConformer Hybrid Large PC (RNNT)",
        adapter_cls=NeMoASRAdapter,
        init_kwargs={
            "model_id": "nvidia/stt_ru_fastconformer_hybrid_large_pc",
            "decoder_type": "rnnt",
            "chunk_long_audio": True,
            "chunk_length_s": 22.0,
            "overlap_s": 0.25,
            "long_audio_threshold_s": 24.5,
        },
        enabled=True,
    ),
    "nvidia_fastconformer_hybrid_large_pc_ctc": ModelSpec(
        key="nvidia_fastconformer_hybrid_large_pc_ctc",
        label="NVIDIA FastConformer Hybrid Large PC (CTC)",
        adapter_cls=NeMoASRAdapter,
        init_kwargs={
            "model_id": "nvidia/stt_ru_fastconformer_hybrid_large_pc",
            "decoder_type": "ctc",
            "chunk_long_audio": True,
            "chunk_length_s": 22.0,
            "overlap_s": 0.25,
            "long_audio_threshold_s": 24.5,
        },
        enabled=True,
    ),
    "gigaam_v3_e2e_ctc": ModelSpec(
        key="gigaam_v3_e2e_ctc",
        label="ai-sage/GigaAM-v3 (e2e_ctc)",
        adapter_cls=GigaAMAdapter,
        init_kwargs={
            "revision": "e2e_ctc",
            "chunk_long_audio": True,
            "chunk_length_s": 22.0,
            "overlap_s": 0.25,
            "long_audio_threshold_s": 24.5,
        },
        enabled=True,
    ),
    "gigaam_v3_e2e_rnnt": ModelSpec(
        key="gigaam_v3_e2e_rnnt",
        label="ai-sage/GigaAM-v3 (e2e_rnnt)",
        adapter_cls=GigaAMAdapter,
        init_kwargs={
            "revision": "e2e_rnnt",
            "chunk_long_audio": True,
            "chunk_length_s": 22.0,
            "overlap_s": 0.25,
            "long_audio_threshold_s": 24.5,
        },
        enabled=True,
    ),
}

selected_specs = [spec for spec in MODEL_REGISTRY.values() if spec.enabled]
print("Выбраны модели:")
for spec in selected_specs:
    print(f" - {spec.label} ({spec.key})")

### Как добавить новую модель

#### Вариант 1 — обычная Hugging Face модель через `pipeline`


In [ ]:
# Пример: добавление новой модели без написания нового класса.
#
# MODEL_REGISTRY["my_new_ctc_model"] = ModelSpec(
#     key="my_new_ctc_model",
#     label="My New CTC Model",
#     adapter_cls=TransformersPipelineAdapter,
#     init_kwargs={
#         "model_id": "org/model-name",
#         "kind": "ctc",          # или "seq2seq"
#         "chunk_length_s": 20,
#         "stride_length_s": (4, 2),
#         "batch_size": 8,
#     },
#     enabled=True,
# )
#
# Для seq2seq-моделей можно добавить generate_kwargs:
# "generate_kwargs": {"task": "transcribe", "language": "russian"}

#### Вариант 2 — если у модели собственный API

Нужно унаследоваться от `BaseASRAdapter` и реализовать три метода:
- `load()`
- `transcribe(audio_path)`
- `unload()`


## 7. Функции оценки


In [ ]:
def build_empty_summary(spec: ModelSpec, pred_df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "model_key": spec.key,
        "model_label": spec.label,
        "n_total": len(pred_df),
        "n_ok": 0,
        "n_error": len(pred_df),
        "corpus_wer": np.nan,
        "corpus_cer": np.nan,
        "avg_row_wer": np.nan,
        "median_row_wer": np.nan,
        "avg_row_cer": np.nan,
        "median_row_cer": np.nan,
        "total_audio_sec": np.nan,
        "total_latency_sec": pred_df["latency_sec"].sum(),
        "avg_sec_per_audio_min": np.nan,
    }


def compute_corpus_wer(refs: List[str], hyps: List[str]) -> float:
    corpus_words = process_words(refs, hyps)
    return float(corpus_words.wer)


def compute_corpus_cer(refs: List[str], hyps: List[str]) -> float:
    joined_ref = " ".join(refs).strip()
    joined_hyp = " ".join(hyps).strip()
    return safe_cer(joined_ref, joined_hyp)


def build_summary(spec: ModelSpec, pred_df: pd.DataFrame) -> Dict[str, Any]:
    ok_df = pred_df[pred_df["status"] == "ok"].copy()

    if ok_df.empty:
        return build_empty_summary(spec, pred_df)

    refs = ok_df["reference_norm"].fillna("").tolist()
    hyps = ok_df["prediction_norm"].fillna("").tolist()

    corpus_wer_value = compute_corpus_wer(refs, hyps)
    corpus_cer_value = compute_corpus_cer(refs, hyps)

    total_audio_sec = ok_df["audio_duration_s"].fillna(0).sum()
    total_latency_sec = ok_df["latency_sec"].sum()

    avg_sec_per_audio_min = (
        (total_latency_sec / total_audio_sec) * 60 if total_audio_sec > 0 else np.nan
    )

    return {
        "model_key": spec.key,
        "model_label": spec.label,
        "n_total": len(pred_df),
        "n_ok": len(ok_df),
        "n_error": len(pred_df) - len(ok_df),
        "corpus_wer": corpus_wer_value,
        "corpus_cer": corpus_cer_value,
        "avg_row_wer": ok_df["row_wer"].mean(),
        "median_row_wer": ok_df["row_wer"].median(),
        "avg_row_cer": ok_df["row_cer"].mean(),
        "median_row_cer": ok_df["row_cer"].median(),
        "total_audio_sec": total_audio_sec,
        "total_latency_sec": total_latency_sec,
        "avg_sec_per_audio_min": avg_sec_per_audio_min,
    }

In [ ]:
def evaluate_model(
    spec: ModelSpec, eval_df: pd.DataFrame
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    adapter = spec.adapter_cls(spec)
    adapter.load()

    rows: List[Dict[str, Any]] = []

    try:
        iterator = tqdm(
            eval_df.itertuples(index=False),
            total=len(eval_df),
            desc=spec.label,
        )

        for row in iterator:
            audio_path = getattr(row, "audio_path")
            ref_raw = getattr(row, "reference_raw")
            ref_clean = getattr(row, "reference_clean")
            ref_norm = getattr(row, "reference_norm")
            audio_duration_s = getattr(row, "audio_duration_s")
            sample_id = getattr(row, ID_COLUMN)

            started_at = time.perf_counter()
            error_message = None
            pred_raw = ""

            try:
                pred_raw = adapter.transcribe(audio_path)
            except Exception as exc:
                error_message = f"{type(exc).__name__}: {exc}"

            latency_sec = time.perf_counter() - started_at
            pred_norm = normalize_text_for_metrics(pred_raw)

            rows.append(
                {
                    "sample_id": sample_id,
                    "model_key": spec.key,
                    "model_label": spec.label,
                    "audio_path": audio_path,
                    "audio_duration_s": audio_duration_s,
                    "reference_raw": ref_raw,
                    "reference_clean": ref_clean,
                    "reference_norm": ref_norm,
                    "prediction_raw": pred_raw,
                    "prediction_norm": pred_norm,
                    "row_wer": safe_wer(ref_norm, pred_norm),
                    "row_cer": safe_cer(ref_norm, pred_norm),
                    "latency_sec": latency_sec,
                    "status": "ok" if error_message is None else "error",
                    "error_message": error_message,
                }
            )
    finally:
        adapter.unload()

    pred_df = pd.DataFrame(rows)
    summary = build_summary(spec, pred_df)

    return summary, pred_df

## 8. Запуск эксперимента


In [ ]:
all_predictions: Dict[str, pd.DataFrame] = {}

In [ ]:
for spec in selected_specs:
    print(f"===== {spec.label} =====")

    pred_path = PREDICTIONS_DIR / f"{spec.key}.parquet"

    if pred_path.exists():
        print(f"Пропуск: результат уже существует -> {pred_path}")
        continue

    summary, pred_df = evaluate_model(spec, df)

    all_predictions[spec.key] = pred_df

    pred_df.to_parquet(pred_path, index=False)

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"Сохранены предсказания: {pred_path}")

In [ ]:
for spec in selected_specs:
    pred_path = PREDICTIONS_DIR / f"{spec.key}.parquet"

    if not pred_path.exists():
        print(f"Пропуск summary: нет файла {pred_path}")
        continue

    pred_df = pd.read_parquet(pred_path)

    all_predictions[spec.key] = pred_df

## 9. Лидерборд


In [ ]:
all_summaries: List[Dict[str, Any]] = []

In [ ]:
for spec in selected_specs:
    pred_path = PREDICTIONS_DIR / f"{spec.key}.parquet"

    if not pred_path.exists():
        print(f"Пропуск summary: нет файла {pred_path}")
        continue

    pred_df = pd.read_parquet(pred_path)
    summary = build_summary(spec, pred_df)
    all_summaries.append(summary)
    print(f"Собран summary для {spec.label}")

leaderboard = (
    pd.DataFrame(all_summaries)
    .sort_values(
        by=["corpus_wer", "corpus_cer", "avg_sec_per_audio_min"],
        ascending=[True, True, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

leaderboard_path = ARTIFACTS_DIR / "leaderboard2.csv"
leaderboard.to_csv(leaderboard_path, index=False)

display(leaderboard)
print(f"Лидерборд сохранён: {leaderboard_path}")

## 10. Визуализация результатов


In [ ]:
metric_columns = [
    "corpus_wer",
    "corpus_cer",
    "avg_sec_per_audio_min",
]

metric_meta = {
    "corpus_wer": {
        "title": "Сравнение моделей по Corpus WER",
        "xlabel": "WER, %",
        "description": "Corpus WER — доля ошибок на уровне слов по всему тестовому корпусу. Чем ниже значение, тем лучше качество распознавания.",
        "is_percent": True,
    },
    "corpus_cer": {
        "title": "Сравнение моделей по Corpus CER",
        "xlabel": "CER, %",
        "description": "Corpus CER — доля ошибок на уровне символов по всему тестовому корпусу. Чем ниже значение, тем лучше качество распознавания.",
        "is_percent": True,
    },
    "avg_sec_per_audio_min": {
        "title": "Среднее время обработки 1 минуты аудио",
        "xlabel": "Секунды",
        "description": "Метрика показывает, сколько секунд требуется модели для обработки 1 минуты аудио. Чем ниже значение, тем выше скорость работы модели.",
        "is_percent": False,
    },
}

plot_df = leaderboard.copy()

available_metrics = [col for col in metric_columns if col in plot_df.columns]

if plot_df.empty or not available_metrics:
    print("Нет корректных результатов для построения графиков.")
else:
    plot_df = plot_df.dropna(subset=["model_label"]).copy()

    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["font.size"] = 11

    for metric in available_metrics:
        metric_df = plot_df.dropna(subset=[metric]).copy()

        if metric_df.empty:
            print(f"Нет корректных значений для метрики: {metric}")
            continue

        meta = metric_meta[metric]

        metric_df = metric_df.sort_values(metric, ascending=True).reset_index(drop=True)

        metric_df = metric_df.iloc[::-1].reset_index(drop=True)

        labels = metric_df["model_label"].astype(str).tolist()
        values = metric_df[metric].astype(float).tolist()

        colors = ["lightgray"] * len(metric_df)
        colors[-1] = "steelblue"

        fig_height = max(5, len(metric_df) * 0.65)
        fig, ax = plt.subplots(figsize=(12, fig_height))

        bars = ax.barh(
            labels,
            values,
            color=colors,
            edgecolor="black",
            alpha=0.9,
        )

        ax.set_title(meta["title"], fontsize=15, pad=12)
        ax.set_xlabel(meta["xlabel"], fontsize=12)
        ax.set_ylabel("Модель", fontsize=12)

        ax.grid(axis="x", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)

        max_val = max(values) if values else 0
        offset = max_val * 0.015 if max_val > 0 else 0.01

        for bar, value in zip(bars, values):
            if meta["is_percent"]:
                text_value = f"{value * 100:.2f}%"
            else:
                text_value = f"{value:.2f}"

            ax.text(
                value + offset,
                bar.get_y() + bar.get_height() / 2,
                text_value,
                va="center",
                ha="left",
                fontsize=10,
            )

        right_limit = max_val * 1.2 if max_val > 0 else 1
        ax.set_xlim(0, right_limit)

        fig.text(
            0.01,
            0.01,
            meta["description"],
            ha="left",
            va="bottom",
            fontsize=10,
        )

        plt.tight_layout(rect=[0, 0.06, 1, 1])
        plt.show()

## 11. Анализ ошибок

Ниже можно посмотреть худшие кейсы по каждой модели и вручную сравнить референс с предсказанием.


In [ ]:
TOP_K_ERRORS = 10

for model_key, pred_df in all_predictions.items():
    print(f"\n### {model_key}")
    view_df = pred_df[pred_df["status"] == "ok"].sort_values(
        "row_wer", ascending=False
    )[
        [
            "sample_id",
            "audio_path",
            "audio_duration_s",
            "row_wer",
            "row_cer",
            "reference_clean",
            "prediction_raw",
        ]
    ]
    display(view_df)